# DeepEval Evaluation Report Template

This notebook is a guided shell for comparing FT vs RAG across Expert and Operational scenarios using CSV outputs downloaded from the admin evaluation endpoint.

## Purpose
Use this notebook to load multiple evaluation runs, normalize the output schema, build summary tables, and write a final analysis and conclusion.

## Supported input contract

The notebook expects DeepEval endpoint outputs, not RAGAS artifacts.

### Checklist
- [ ] Each run is labeled with a system name and scenario name.
- [ ] Result files contain question, answer, expected output, and metric columns.
- [ ] Golden files are versioned and documented.
- [ ] Missing files are reported before any analysis starts.

## DeepEval endpoint schema

This template is aligned with the admin evaluation endpoint output shape, not with RAGAS artifacts.

### Expected fields
- `Question`
- `Actual Answer`
- `Expected Output`
- `* Score` metric columns
- `* Reason` metric columns
- `latency_s` for trade-off analysis

### Checklist
- [ ] The template matches the endpoint export contract.
- [ ] The notebook remains usable with real CSV downloads.
- [ ] Metric reasons are kept available for qualitative review.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

sns.set_theme(style='whitegrid')
pd.set_option('display.max_colwidth', 120)

ENDPOINT_CORE_COLUMNS = [
    'Question',
    'Actual Answer',
    'Expected Output',
]

METRIC_SCORE_COLUMNS = [
    'Correctness Score',
    'Faithfulness Score',
    'Answer Relevancy Score',
    'ContextualPrecision Score',
    'ContextualRecall Score',
    'ContextualRelevancy Score',
]

METRIC_REASON_COLUMNS = [
    'Correctness Reason',
    'Faithfulness Reason',
    'Answer Relevancy Reason',
    'ContextualPrecision Reason',
    'ContextualRecall Reason',
    'ContextualRelevancy Reason',
]

def load_table(path):
    path = Path(path)
    if not path.exists():
        return pd.DataFrame()
    if path.suffix.lower() == '.csv':
        return pd.read_csv(path)
    if path.suffix.lower() == '.json':
        with path.open('r', encoding='utf-8') as handle:
            data = json.load(handle)
        if isinstance(data, list):
            return pd.DataFrame(data)
        return pd.DataFrame([data])
    raise ValueError(f'Unsupported file type: {path.suffix}')

def normalize_endpoint_export(df):
    if df.empty:
        return df
    rename_map = {
        'question': 'Question',
        'query': 'Question',
        'actual_answer': 'Actual Answer',
        'answer': 'Actual Answer',
        'result': 'Actual Answer',
        'expected_output': 'Expected Output',
        'reference': 'Expected Output',
        'latency': 'latency_s',
    }
    return df.rename(columns={source: target for source, target in rename_map.items() if source in df.columns})

def validate_endpoint_export(df):
    required_columns = ENDPOINT_CORE_COLUMNS + METRIC_SCORE_COLUMNS
    missing_columns = [column for column in required_columns if column not in df.columns]
    return {
        'valid': not missing_columns,
        'missing': missing_columns,
    }

def load_evaluation_runs(source_map):
    frames = []
    for run_name, file_path in source_map.items():
        frame = load_table(file_path)
        if frame.empty:
            print(f'Skipping missing or empty file for {run_name}: {file_path}')
            continue
        frame = normalize_endpoint_export(frame.copy())
        frame['run_name'] = run_name
        if '-' in run_name:
            system_type, scenario = run_name.split('-', 1)
        else:
            system_type, scenario = run_name, 'Unknown'
        frame['system_type'] = system_type
        frame['scenario'] = scenario
        frames.append(frame)
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)

def detect_metric_columns(df):
    return [column for column in df.columns if column.endswith(' Score')] if not df.empty else []

def build_summary_table(df):
    metric_columns = detect_metric_columns(df)
    if df.empty or not metric_columns:
        return pd.DataFrame()
    group_columns = ['system_type', 'scenario']
    numeric_columns = metric_columns + (['latency_s'] if 'latency_s' in df.columns else [])
    return df.groupby(group_columns, as_index=False)[numeric_columns].mean()

def build_wins_table(summary_df, metric_columns):
    if summary_df.empty or not metric_columns:
        return pd.DataFrame()
    rows = []
    for scenario_name in summary_df['scenario'].dropna().unique():
        subset = summary_df[summary_df['scenario'] == scenario_name]
        for metric in metric_columns:
            best_row = subset.loc[subset[metric].idxmax()]
            rows.append({
                'scenario': scenario_name,
                'metric': metric,
                'winner': best_row['system_type'],
                'score': best_row[metric],
            })
    return pd.DataFrame(rows)

def build_reason_table(df, max_rows=8):
    reason_columns = [column for column in df.columns if column.endswith(' Reason')]
    if df.empty or not reason_columns:
        return pd.DataFrame()
    preview_columns = ['system_type', 'scenario', 'Question'] + reason_columns
    return df[preview_columns].head(max_rows)

def plot_metric_bars(summary_df, metric_columns):
    melted = summary_df.melt(id_vars=['system_type', 'scenario'], value_vars=metric_columns, var_name='metric', value_name='score')
    g = sns.catplot(
        data=melted,
        kind='bar',
        x='metric',
        y='score',
        hue='system_type',
        col='scenario',
        palette='Set2',
        height=4.5,
        aspect=1.3,
        sharey=True,
    )
    g.set_xticklabels(rotation=45, ha='right')
    g.set_titles('{col_name}')
    g.fig.suptitle('DeepEval metric comparison by system and scenario', y=1.05)
    plt.show()

def plot_metric_distributions(df, metric_columns):
    melted = df.melt(id_vars=['system_type', 'scenario'], value_vars=metric_columns, var_name='metric', value_name='score')
    fig, ax = plt.subplots(figsize=(12, 5))
    sns.boxplot(data=melted, x='metric', y='score', hue='system_type', ax=ax, palette='Set3')
    ax.set_title('Score dispersion by metric')
    ax.tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.show()

def plot_latency_scatter(df):
    metric_columns = detect_metric_columns(df)
    if df.empty or 'latency_s' not in df.columns or not metric_columns:
        print('Latency is not available in the current dataset.')
        return
    df_plot = df.copy()
    df_plot['quality_mean'] = df_plot[metric_columns].mean(axis=1)
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.scatterplot(data=df_plot, x='latency_s', y='quality_mean', hue='system_type', style='scenario', s=90, ax=ax)
    ax.set_title('Latency vs average quality')
    ax.set_xlabel('Latency (s)')
    ax.set_ylabel('Average quality score')
    plt.tight_layout()
    plt.show()

## 1. Load and normalize evaluation outputs

Place the CSV or JSON exports downloaded from `/api/admin/evaluate/download/{task_id}` in the paths declared below.

### Checklist
- [ ] Every run has an explicit system label.
- [ ] Every run has an explicit scenario label.
- [ ] File paths are documented and reproducible.
- [ ] The notebook reports missing files instead of failing silently.

In [ ]:
RESULT_SOURCES = {
    'FT-Expert': 'path/to/ft_expert_results.csv',
    'RAG-Expert': 'path/to/rag_expert_results.csv',
    'FT-Operational': 'path/to/ft_operational_results.csv',
    'RAG-Operational': 'path/to/rag_operational_results.csv',
}

GOLDEN_SOURCES = {
    'Expert': 'path/to/golden_expert.json',
    'Operational': 'path/to/golden_operational.json',
}

all_results_df = load_evaluation_runs(RESULT_SOURCES)
golden_info_df = load_evaluation_runs(GOLDEN_SOURCES)

print(f"Loaded rows: {len(all_results_df)}")
display(all_results_df.head())

## 2. Validate schema and metric coverage

Standardize the output so every run can be compared under the same reporting rules.

### Checklist
- [ ] Required core columns exist.
- [ ] Metric columns are numeric and bounded between 0 and 1.
- [ ] Optional latency is either present or explicitly marked as missing.
- [ ] Metric reason columns are available for qualitative review.
- [ ] The schema remains stable across all runs.

In [ ]:
def validate_results_schema(df):
    if df.empty:
        return {"status": "empty", "missing": []}
    required = ["question", "actual_answer", "expected_output", "system_type", "scenario"]
    missing = [column for column in required if column not in df.columns]
    return {"status": "ok" if not missing else "missing", "missing": missing}

schema_status = validate_results_schema(all_results_df)
print(schema_status)

metric_columns = detect_metric_columns(all_results_df)
summary_df = build_summary_table(all_results_df)
display(summary_df)

## 3. Metric reason review

The endpoint export includes `* Reason` columns. This section shows how to review the qualitative evidence behind the scores.

### Checklist
- [ ] Metric reasons are available for both FT and RAG.
- [ ] The report can quote reasons directly when discussing trade-offs.
- [ ] The analysis keeps quantitative and qualitative evidence together.

In [ ]:
reason_snapshot_df = build_reason_table(all_results_df)

if reason_snapshot_df.empty:
    print('No metric reasons are available in the current dataset.')
else:
    display(reason_snapshot_df)

    scenario_reason_summary = (
        reason_snapshot_df.groupby(['system_type', 'scenario'], as_index=False)
        .first()
    )
    display(scenario_reason_summary)

## 4. KPI tables

Use the aggregate tables to compare systems and scenarios side by side.

### Checklist
- [ ] Mean scores are reported by system and scenario.
- [ ] A winner table exists for each metric.
- [ ] The analysis distinguishes quality metrics from latency.
- [ ] Tables are easy to paste into the final report.

In [ ]:
wins_df = build_wins_table(summary_df, metric_columns)
display(wins_df)

if "latency_s" in summary_df.columns:
    display(summary_df[["system_type", "scenario", "latency_s"]])

## 5. Visual analytics

Create plots that make the FT vs RAG trade-offs visible.

### Checklist
- [ ] Bar charts compare mean metric scores.
- [ ] Boxplots show score dispersion when multiple rows exist.
- [ ] Latency is compared against quality where available.
- [ ] The figure set supports a direct story in the final report.

In [ ]:
def plot_metric_bars(summary_df, metric_columns):
    if summary_df.empty or not metric_columns:
        print("No data available for metric bars yet.")
        return
    melted = summary_df.melt(id_vars=["system_type", "scenario"], value_vars=metric_columns, var_name="metric", value_name="score")
    g = sns.catplot(
        data=melted,
        kind="bar",
        x="metric",
        y="score",
        hue="system_type",
        col="scenario",
        palette="Set2",
        height=4.5,
        aspect=1.3,
        sharey=True,
    )
    g.set_xticklabels(rotation=45, ha="right")
    g.set_titles("{col_name}")
    g.fig.suptitle("DeepEval metric comparison by system and scenario", y=1.05)
    plt.show()

def plot_metric_distributions(df, metric_columns):
    if df.empty or not metric_columns:
        print("No data available for distribution plots yet.")
        return
    melted = df.melt(id_vars=["system_type", "scenario"], value_vars=metric_columns, var_name="metric", value_name="score")
    fig, ax = plt.subplots(figsize=(12, 5))
    sns.boxplot(data=melted, x="metric", y="score", hue="system_type", ax=ax, palette="Set3")
    ax.set_title("Score dispersion by metric")
    ax.tick_params(axis="x", rotation=45)
    plt.tight_layout()
    plt.show()

def plot_latency_scatter(df):
    if df.empty or "latency_s" not in df.columns:
        print("Latency is not available in the current dataset.")
        return
    quality_columns = [column for column in detect_metric_columns(df) if column != "latency_s"]
    df_plot = df.copy()
    df_plot["quality_mean"] = df_plot[quality_columns].mean(axis=1)
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.scatterplot(data=df_plot, x="latency_s", y="quality_mean", hue="system_type", style="scenario", s=90, ax=ax)
    ax.set_title("Latency vs average quality")
    ax.set_xlabel("Latency (s)")
    ax.set_ylabel("Average quality score")
    plt.tight_layout()
    plt.show()

## 6. Comparative analysis

Write the interpretation directly from the tables and charts.

### Checklist
- [ ] State which system wins each scenario.
- [ ] Explain whether gains are consistent or metric-specific.
- [ ] Comment on latency-quality trade-offs.
- [ ] Note failure modes and limitations.

In [ ]:
analysis_prompts = [
    "Which system wins the Expert scenario on correctness and why?",
    "Which system wins the Operational scenario on latency and answer relevancy?",
    "Are there metrics where the winner changes by scenario?",
    "Do any scores suggest risk of hallucination or weak grounding?",
]

display(Markdown("Use the prompts below to draft the narrative analysis from the tables above."))
for prompt in analysis_prompts:
    print(f"- {prompt}")

## 7. Final conclusion

Use the evidence above to write the final conclusion for the report.

### Checklist
- [ ] The conclusion summarizes the main findings by scenario.
- [ ] The conclusion separates evidence from interpretation.
- [ ] The conclusion states the limitations of the evaluation.
- [ ] The conclusion ends with a practical recommendation.

### Conclusion placeholder
Replace this text with the final report summary once the real endpoint results are loaded.